# Preparing a Model for Deployment: The Full Save/Load/Predict Cycle

Deployment is not just saving a model file. A real deployment artifact is a bundle: the trained pipeline, the preprocessing transformations, and a metadata file describing what the model is and what it expects. This notebook builds that bundle from scratch and verifies it works correctly after loading.

**Learning objectives**
1. Understand what files a deployment artifact must contain beyond just the model weights.
2. Build a `sklearn.Pipeline` that combines preprocessing and a classifier in one object.
3. Save the pipeline with joblib, delete it from memory, reload it, and verify predictions are identical.
4. Save a metadata JSON alongside the model and write a `load_model_bundle()` function that validates it.


## 1  What does a deployment artifact need?

A model in production needs three things:

| File | Purpose |
|------|--------|
| `pipeline.joblib` | The trained model + all preprocessing steps |
| `metadata.json` | Version, feature names, accuracy, training date |
| `requirements.txt` | Exact package versions the model was trained with |

If you save the model but not the scaler, predictions will be wrong. If you save the model but not the metadata, you have no way to check whether the file on the server is the one you intended to deploy.


In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np
import joblib
import json
import os
from datetime import datetime

iris = load_iris()
FEATURE_NAMES = list(iris.feature_names)
CLASS_NAMES = list(iris.target_names)

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

print("Feature names:", FEATURE_NAMES)
print("Class names  :", CLASS_NAMES)
print(f"Training samples: {len(X_train)}   Test samples: {len(X_test)}")


Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Class names  : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Training samples: 120   Test samples: 30


## 2  Build a preprocessing pipeline

Wrapping the scaler and classifier in a `Pipeline` means you only have to call `pipeline.predict(raw_features)` — the pipeline applies the scaler automatically. This is what you should always save: the whole pipeline, not just the classifier.


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)

# Reference predictions — we'll verify the loaded pipeline produces the same values
reference_preds = pipeline.predict(X_test)

print(f"Pipeline accuracy: {accuracy:.2%}")
print(f"Steps in pipeline: {[s[0] for s in pipeline.steps]}")
print(f"Scaler mean (first feature): {pipeline.named_steps['scaler'].mean_[0]:.4f}")


Pipeline accuracy: 100.00%
Steps in pipeline: ['scaler', 'classifier']
Scaler mean (first feature): 5.8092


## 3  Save the pipeline with joblib


In [3]:
BUNDLE_DIR = "/tmp/iris_model_bundle"
os.makedirs(BUNDLE_DIR, exist_ok=True)

PIPELINE_PATH = os.path.join(BUNDLE_DIR, "pipeline.joblib")
joblib.dump(pipeline, PIPELINE_PATH)

size_kb = os.path.getsize(PIPELINE_PATH) / 1024
print(f"Pipeline saved to: {PIPELINE_PATH}")
print(f"File size: {size_kb:.1f} KB")


Pipeline saved to: /tmp/iris_model_bundle/pipeline.joblib
File size: 183.1 KB


## 4  Load fresh and verify identical predictions

We delete the original object from memory (simulating a fresh server), reload from disk, and confirm predictions are bit-for-bit identical to the reference.


In [4]:
# Delete the original pipeline to simulate a fresh environment
del pipeline

# Reload from disk
loaded_pipeline = joblib.load(PIPELINE_PATH)

loaded_preds = loaded_pipeline.predict(X_test)
match = np.array_equal(loaded_preds, reference_preds)

print(f"Predictions match original: {match}")
print(f"Loaded pipeline accuracy  : {loaded_pipeline.score(X_test, y_test):.2%}")
print(f"Scaler mean still intact  : {loaded_pipeline.named_steps['scaler'].mean_[0]:.4f}")

# Confirm the scaler is embedded — raw features go in, correct predictions come out
sample_raw = X_test[:3]
print(f"\nSample predictions on raw features (scaler applied inside pipeline):")
for i, pred in enumerate(loaded_pipeline.predict(sample_raw)):
    print(f"  Sample {i}: {CLASS_NAMES[pred]}")


Predictions match original: True
Loaded pipeline accuracy  : 100.00%
Scaler mean still intact  : 5.8092

Sample predictions on raw features (scaler applied inside pipeline):
  Sample 0: versicolor
  Sample 1: setosa
  Sample 2: virginica


## 5  Save model metadata as JSON

The metadata file records what this model is, what it was trained on, and how well it performed. Without this file, you cannot tell which version of the model is running in production.


In [5]:
import sklearn

metadata = {
    "model_name": "iris-classifier",
    "version": "1.0.0",
    "training_date": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    "feature_names": FEATURE_NAMES,
    "class_names": list(CLASS_NAMES),
    "n_features": len(FEATURE_NAMES),
    "n_classes": len(CLASS_NAMES),
    "test_accuracy": round(loaded_pipeline.score(X_test, y_test), 4),
    "sklearn_version": sklearn.__version__,
    "pipeline_steps": [s[0] for s in loaded_pipeline.steps],
}

METADATA_PATH = os.path.join(BUNDLE_DIR, "metadata.json")
with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved:")
print(json.dumps(metadata, indent=2))


Metadata saved:
{
  "model_name": "iris-classifier",
  "version": "1.0.0",
  "training_date": "2026-08-20T12:19:03Z",
  "feature_names": [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
  ],
  "class_names": [
    "setosa",
    "versicolor",
    "virginica"
  ],
  "n_features": 4,
  "n_classes": 3,
  "test_accuracy": 1.0,
  "sklearn_version": "1.8.0",
  "pipeline_steps": [
    "scaler",
    "classifier"
  ]
}


/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_44196/1433353957.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "training_date": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),


## 6  Load the full model bundle

A deployment system should load both files together and validate that the metadata is consistent before serving any traffic.


In [6]:
def load_model_bundle(bundle_dir):
    """Load a model pipeline and its metadata, with basic validation."""
    pipeline_path = os.path.join(bundle_dir, "pipeline.joblib")
    metadata_path = os.path.join(bundle_dir, "metadata.json")

    if not os.path.exists(pipeline_path):
        raise FileNotFoundError(f"pipeline.joblib not found in {bundle_dir}")
    if not os.path.exists(metadata_path):
        raise FileNotFoundError(f"metadata.json not found in {bundle_dir}")

    model = joblib.load(pipeline_path)
    with open(metadata_path) as f:
        meta = json.load(f)

    # Validate that the model expects the right number of features
    expected_features = meta["n_features"]
    scaler = model.named_steps["scaler"]
    actual_features = scaler.n_features_in_
    if expected_features != actual_features:
        raise ValueError(
            f"Metadata says {expected_features} features, "
            f"but model has {actual_features}"
        )

    return model, meta


# Load the bundle
model, meta = load_model_bundle(BUNDLE_DIR)

print(f"Loaded model    : {meta['model_name']} v{meta['version']}")
print(f"Trained on      : {meta['training_date']}")
print(f"Feature names   : {meta['feature_names']}")
print(f"Test accuracy   : {meta['test_accuracy']:.2%}")

# Confirm predictions still match
bundle_preds = model.predict(X_test)
print(f"\nPredictions match reference: {np.array_equal(bundle_preds, reference_preds)}")


Loaded model    : iris-classifier v1.0.0
Trained on      : 2026-08-20T12:19:03Z
Feature names   : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Test accuracy   : 100.00%

Predictions match reference: True


## Summary

A deployment artifact is a bundle, not just a model file. The three required components are:

| Component | Why it matters |
|---|---|
| `pipeline.joblib` (with scaler embedded) | The model and all preprocessing in one object — raw input goes in, prediction comes out |
| `metadata.json` | Records version, features, accuracy — makes deployments auditable |
| Verification step | Confirms the loaded model gives the same predictions as the original |

Saving the classifier without the scaler is a common mistake that causes incorrect predictions in production because the input distribution at inference time won't match what the model was trained on.


## Self-check

1. **What goes wrong if you save only the classifier but not the scaler?** Run `model.predict(X_test)` after removing the scaler step and observe the difference in predictions.
2. **What should you store in the metadata JSON?** Look at the metadata dict in Section 5 — which fields would you need to debug a model behaving unexpectedly in production?
3. **How would you check that the loaded model gives the same predictions as the original?** Look at the `np.array_equal(loaded_preds, reference_preds)` check in Section 4 — what would a `False` result tell you?
